In [2]:
!pip install timm

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
     |████████████████████████████████| 2.5 MB 3.4 MB/s eta 0:00:01
You should consider upgrading via the '/usr/bin/python -m pip install --upgrade pip' command.


In [13]:
import os
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from tqdm import tqdm
from PIL import Image
import timm
from transformers import (
    AutoTokenizer,
    AutoModel,
    get_linear_schedule_with_warmup
)
from torch.cuda.amp import autocast, GradScaler
from sklearn.model_selection import train_test_split
import re

In [30]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from tqdm import tqdm
from PIL import Image
from sklearn.model_selection import train_test_split
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoImageProcessor,
    AutoModelForImageClassification,
    get_linear_schedule_with_warmup
)
from torch.cuda.amp import autocast, GradScaler
from sklearn.preprocessing import OneHotEncoder, StandardScaler


In [41]:
# =====================================================================================
# Configuration - Tuned for Faster Iteration
# =====================================================================================
class Config:
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    BATCH_SIZE = 64
    NUM_WORKERS = 8
    TRAIN_SUBSET_FRACTION = 1.0
    TEST_SUBSET_FRACTION = 1.0
    EPOCHS = 10

    # --- MODIFICATION #2: ADJUST LEARNING RATE ---
    # Smaller models often train better with a slightly higher learning rate.
    LR = 3e-5 # Increased from 1e-6 for faster learning with smaller models
    # ---------------------------------------------

    MAX_LEN = 128

    # --- MODIFICATION #3: SWAP THE MODELS ---
    # Replace RoBERTa with the smaller, faster DistilBERT.
    TEXT_MODEL = "distilbert-base-uncased"
    # Replace CLIP's Vision Transformer with the much smaller and faster EfficientNet.
    IMAGE_MODEL_NAME = "google/vit-base-patch16-224-in21k"
    # ---------------------------------------

    TRAIN_IMAGE_DIR = "../images_train"
    TEST_IMAGE_DIR = "../images_test"
    SEED = 42
    CHECKPOINT_DIR = "./checkpoints_light" # New directory for this faster run
    LATEST_CHECKPOINT_PATH = os.path.join(CHECKPOINT_DIR, "latest_checkpoint.pth")

print(f"Using device: {Config.DEVICE}")
print(f"Batch Size: {Config.BATCH_SIZE}, Num Workers: {Config.NUM_WORKERS}")
print(f"Learning Rate: {Config.LR}")
os.makedirs(Config.CHECKPOINT_DIR, exist_ok=True)


Using device: cuda
Batch Size: 64, Num Workers: 8
Learning Rate: 3e-05


In [4]:
DATASET_FOLDER = '../dataset/'

# Load the datasets
train_df = pd.read_csv(DATASET_FOLDER + 'train.csv')
test_df = pd.read_csv(DATASET_FOLDER + 'test.csv')

print("Training data shape:", train_df.shape)
print("Test data shape:", test_df.shape)

Training data shape: (75000, 4)
Test data shape: (75000, 3)


In [5]:
IMAGE_TRAIN_FOLDER = '../images_train/' 
train_df['filename'] = train_df['image_link'].str.split('/').str[-1]
train_df.head()

,sample_id,catalog_content,image_link,price,filename
0,33127,"Item Name: La Victoria Green Taco Sauce Mild, ...",https://m.media-amazon.com/images/I/51mo8htwTH...,4.89,51mo8htwTHL.jpg
1,198967,"Item Name: Salerno Cookies, The Original Butte...",https://m.media-amazon.com/images/I/71YtriIHAA...,13.12,71YtriIHAAL.jpg
2,261251,"Item Name: Bear Creek Hearty Soup Bowl, Creamy...",https://m.media-amazon.com/images/I/51+PFEe-w-...,1.97,51+PFEe-w-L.jpg
3,55858,Item Name: Judee’s Blue Cheese Powder 11.25 oz...,https://m.media-amazon.com/images/I/41mu0HAToD...,30.34,41mu0HAToDL.jpg
4,292686,"Item Name: kedem Sherry Cooking Wine, 12.7 Oun...",https://m.media-amazon.com/images/I/41sA037+Qv...,66.49,41sA037+QvL.jpg


In [6]:
train_df['image_path'] = IMAGE_TRAIN_FOLDER + train_df['filename']

In [7]:
IMAGE_TEST_FOLDER = '../images_test/' 
test_df['filename'] = test_df['image_link'].str.split('/').str[-1]
test_df['image_path'] = IMAGE_TEST_FOLDER + test_df['filename']

In [10]:
def parse_catalog_content(text):
    """
    Parses the structured text from catalog_content into separate features.
    """
    # Use a dictionary to store our extracted features
    features = {
        'item_name': '',
        'bullet_points': '',
        'value': np.nan, # Use NaN (Not a Number) for missing numerical values
        'unit': 'unknown'
    }

    # Ensure the input is a string
    text = str(text)

    # --- Extract Item Name ---
    # Look for "Item Name:" followed by any characters until the next "Bullet Point"
    match = re.search(r'Item Name:(.*?)(Bullet Point|$)', text, re.DOTALL)
    if match:
        features['item_name'] = match.group(1).strip()

    # --- Extract all Bullet Points and combine them ---
    # Find all occurrences of "Bullet Point X:"
    bullets = re.findall(r'Bullet Point \d+:(.*?)(Bullet Point|Value|$)', text, re.DOTALL)
    # Join all found bullet points into a single string
    features['bullet_points'] = ' '.join([b[0].strip() for b in bullets])

    # --- Extract Value ---
    match = re.search(r'Value:\s*([\d\.]+)', text)
    if match:
        try:
            # Convert the extracted string to a float
            features['value'] = float(match.group(1))
        except ValueError:
            # If conversion fails, leave it as NaN
            pass 

    # --- Extract Unit ---
    match = re.search(r'Unit:\s*(\w+)', text)
    if match:
        features['unit'] = match.group(1).strip()
        
    return features
def extract_ipq(text):
    """
    Extracts the Item Pack Quantity (IPQ) from a product description string.
    """
    text = str(text).lower()
    patterns = [
        r'pack of (\d+)', r'(\d+)\s*-?\s*pack', r'(\d+)\s*count',
        r'(\d+)\s*ct', r'ipq\s*:\s*(\d+)', r'case of (\d+)'
    ]
    for pattern in patterns:
        match = re.search(pattern, text)
        if match:
            return int(match.group(1))
    return 1 # Default to 1 if no pack size is found

In [14]:
parsed_features_train = train_df['catalog_content'].apply(parse_catalog_content).apply(pd.Series)

parsed_features_test = test_df['catalog_content'].apply(parse_catalog_content).apply(pd.Series)
train_df = pd.concat([train_df, parsed_features_train], axis=1)
test_df = pd.concat([test_df, parsed_features_test], axis=1)
train_df['ipq'] = train_df['catalog_content'].apply(extract_ipq)
test_df['ipq'] = test_df['catalog_content'].apply(extract_ipq)

In [15]:
train_df['clean_text'] = train_df['item_name'] + ' ' + train_df['bullet_points']
test_df['clean_text'] = test_df['item_name'] + ' ' + test_df['bullet_points']
train_df[['catalog_content', 'clean_text']].head()

,catalog_content,clean_text
0,"Item Name: La Victoria Green Taco Sauce Mild, ...","La Victoria Green Taco Sauce Mild, 12 Ounce (P..."
1,"Item Name: Salerno Cookies, The Original Butte...","Salerno Cookies, The Original Butter Cookies, ..."
2,"Item Name: Bear Creek Hearty Soup Bowl, Creamy...","Bear Creek Hearty Soup Bowl, Creamy Chicken wi..."
3,Item Name: Judee’s Blue Cheese Powder 11.25 oz...,Judee’s Blue Cheese Powder 11.25 oz - Gluten-F...
4,"Item Name: kedem Sherry Cooking Wine, 12.7 Oun...","kedem Sherry Cooking Wine, 12.7 Ounce - 12 per..."


In [16]:
train_df_cleaned = train_df.drop(columns=['catalog_content', 'filename', 'item_name', 'bullet_points','image_link'])
test_df_cleaned = test_df.drop(columns=['catalog_content', 'filename', 'item_name', 'bullet_points','image_link'])

In [17]:
train_df_cleaned.head()

,sample_id,price,image_path,value,unit,ipq,clean_text
0,33127,4.89,../images_train/51mo8htwTHL.jpg,72.00,Fl,6,"La Victoria Green Taco Sauce Mild, 12 Ounce (P..."
1,198967,13.12,../images_train/71YtriIHAAL.jpg,32.00,Ounce,4,"Salerno Cookies, The Original Butter Cookies, ..."
2,261251,1.97,../images_train/51+PFEe-w-L.jpg,11.40,Ounce,6,"Bear Creek Hearty Soup Bowl, Creamy Chicken wi..."
3,55858,30.34,../images_train/41mu0HAToDL.jpg,11.25,Ounce,1,Judee’s Blue Cheese Powder 11.25 oz - Gluten-F...
4,292686,66.49,../images_train/41sA037+QvL.jpg,12.00,Count,1,"kedem Sherry Cooking Wine, 12.7 Ounce - 12 per..."


In [18]:
train_df_cleaned['log_price'] = np.log1p(train_df_cleaned['price'])

In [19]:
median_value = train_df_cleaned['value'].median()
median_value_test=test_df_cleaned['value'].median()
# Fill missing values in both train and test sets with this median
train_df_cleaned['value'].fillna(median_value, inplace=True)
test_df_cleaned['value'].fillna(median_value, inplace=True)

In [21]:
train_df_cleaned.columns

Index(['sample_id', 'price', 'image_path', 'value', 'unit', 'ipq',
       'clean_text', 'log_price'],
      dtype='object')

In [69]:
test_df_cleaned.columns

Index(['sample_id', 'image_path', 'value', 'unit', 'ipq', 'clean_text'], dtype='object')

In [60]:
def prepare_data(train_cleaned_df):
    df = train_cleaned_df.copy()

    # --- One-hot encode 'unit' ---
    encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
    encoded_units = encoder.fit_transform(df[["unit"]])
    encoded_df = pd.DataFrame(encoded_units, columns=encoder.get_feature_names_out(["unit"]))
    df = pd.concat([df.drop(columns=["unit"]), encoded_df], axis=1)

    # --- Train/Validation Split ---
    train_df, val_df = train_test_split(df, test_size=0.1, random_state=Config.SEED)

    # --- Scale numeric features ---
    numeric_cols = ["value", "ipq"]
    scaler = StandardScaler()

    train_df[numeric_cols] = scaler.fit_transform(train_df[numeric_cols])
    val_df[numeric_cols] = scaler.transform(val_df[numeric_cols])

    return (
        train_df.reset_index(drop=True),
        val_df.reset_index(drop=True),
        encoder,  # keep for inference encoding
        scaler,   # keep for inference scaling
    )

In [61]:
import torch
from PIL import Image

class ProductDataset(torch.utils.data.Dataset):
    def __init__(self, df, tokenizer, image_processor, numeric_cols, is_train=True):
        self.df = df
        self.tokenizer = tokenizer
        self.processor = image_processor
        self.numeric_cols = numeric_cols  # List of numeric + one-hot columns
        self.is_train = is_train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # --- Text ---
        text = row["clean_text"]  # Only use clean_text if catalog_content not needed
        text_inputs = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=Config.MAX_LEN,
            return_tensors="pt"
        )

        # --- Image ---
        try:
            img = Image.open(row["image_path"]).convert("RGB")
        except Exception:
            img = Image.new("RGB", (224, 224), color=(255, 255, 255))
        img_inputs = self.processor(images=img, return_tensors="pt")

        # --- Numeric (numeric + one-hot) ---
        numeric = torch.tensor(row[self.numeric_cols].to_numpy(dtype=np.float32))

        output = {
            "input_ids": text_inputs["input_ids"].squeeze(),
            "attention_mask": text_inputs["attention_mask"].squeeze(),
            "pixel_values": img_inputs["pixel_values"].squeeze(),
            "numeric": numeric
        }

        if self.is_train:
            output["log_price"] = torch.tensor(row["log_price"], dtype=torch.float32)
            output["price"] = torch.tensor(row["price"], dtype=torch.float32)

        return output



In [70]:
def prepare_inference_data(new_df, encoder, scaler):
    df = new_df.copy()

    # Encode units using trained encoder
    encoded_units = encoder.transform(df[["unit"]])
    encoded_df = pd.DataFrame(encoded_units, columns=encoder.get_feature_names_out(["unit"]))
    df = pd.concat([df.drop(columns=["unit"]), encoded_df], axis=1)

    # Scale numeric features using trained scaler
    numeric_cols = ["value", "ipq"]
    df[numeric_cols] = scaler.transform(df[numeric_cols])

    return df

In [63]:
class MultiModalRegressor(nn.Module):
    def __init__(self, text_encoder, img_encoder):
        super().__init__()
        self.text_encoder = text_encoder
        self.img_encoder = img_encoder
        text_hidden = text_encoder.config.hidden_size
        img_hidden = img_encoder.config.hidden_size

        self.text_proj = nn.Linear(text_hidden, 256)
        self.img_proj = nn.Linear(img_hidden, 256)
        self.num_proj = nn.Linear(85, 64)

        self.head = nn.Sequential(
            nn.Linear(256 + 256 + 64, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, batch):
        # --- Text ---
        txt_out = self.text_encoder(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"]
        ).last_hidden_state.mean(dim=1)

        # --- Image ---
        img_out = self.img_encoder(pixel_values=batch["pixel_values"]).last_hidden_state.mean(dim=1)

        # --- Numeric ---
        num_out = self.num_proj(batch["numeric"])

        # --- Fusion ---
        fused = torch.cat([self.text_proj(txt_out), self.img_proj(img_out), num_out], dim=1)
        return self.head(fused)


In [64]:
def smape(y_true, y_pred):
    valid = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true, y_pred = y_true[valid], y_pred[valid]
    num = np.abs(y_pred - y_true)
    den = (np.abs(y_true) + np.abs(y_pred)) / 2
    return np.mean(num / (den + 1e-8)) * 100


In [65]:
def train_model(train_df, val_df):
    tokenizer = AutoTokenizer.from_pretrained(Config.TEXT_MODEL)
    image_processor = AutoImageProcessor.from_pretrained(Config.IMAGE_MODEL_NAME)

    text_encoder = AutoModel.from_pretrained(Config.TEXT_MODEL)
    img_encoder = AutoModel.from_pretrained(Config.IMAGE_MODEL_NAME)

    model = MultiModalRegressor(text_encoder, img_encoder).to(Config.DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=Config.LR)
    criterion = nn.SmoothL1Loss()
    scaler = GradScaler()
    
    numeric_cols = ['value', 'ipq'] + [c for c in train_df.columns if c.startswith("unit_")]
    train_ds = ProductDataset(train_df, tokenizer, image_processor,numeric_cols, True)
    val_ds = ProductDataset(val_df, tokenizer, image_processor, numeric_cols,True)
    train_loader = torch.utils.data.DataLoader(train_ds, batch_size=Config.BATCH_SIZE, shuffle=True, num_workers=Config.NUM_WORKERS)
    val_loader = torch.utils.data.DataLoader(val_ds, batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=Config.NUM_WORKERS)

    num_steps = len(train_loader) * Config.EPOCHS
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=num_steps)

    best_smape = float('inf')
    for epoch in range(Config.EPOCHS):
        model.train()
        total_loss = 0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{Config.EPOCHS}")
        for batch in pbar:
            y = batch.pop("log_price").to(Config.DEVICE).unsqueeze(1)
            batch.pop("price")
            batch = {k: v.to(Config.DEVICE) for k, v in batch.items()}
            optimizer.zero_grad()
            with autocast():
                preds = model(batch)
                loss = criterion(preds, y)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            total_loss += loss.item()
            pbar.set_postfix(loss=f"{loss.item():.4f}")

        avg_loss = total_loss / len(train_loader)
        model.eval()
        preds_all, trues_all = [], []
        with torch.no_grad():
            for batch in val_loader:
                y_true = batch.pop("price").numpy()
                batch.pop("log_price")
                batch = {k: v.to(Config.DEVICE) for k, v in batch.items()}
                with autocast():
                    preds = model(batch).cpu().squeeze().numpy()
                preds = np.clip(preds, None, 20)
                preds_all.append(np.expm1(preds))
                trues_all.append(y_true)
        preds_all = np.concatenate(preds_all)
        trues_all = np.concatenate(trues_all)
        val_smape = smape(trues_all, preds_all)
        print(f"Epoch {epoch+1}: TrainLoss={avg_loss:.4f} | ValSMAPE={val_smape:.2f}%")

        if val_smape < best_smape:
            best_smape = val_smape
            torch.save(model.state_dict(), os.path.join(Config.CHECKPOINT_DIR, f"best_model_{val_smape:.2f}.pth"))
            print(f"✅ Saved new best model with SMAPE={val_smape:.2f}%")

    print(f"\nTraining done. Best SMAPE={best_smape:.2f}%")
    return model


In [66]:
train_df, val_df,encoder,scaler = prepare_data(train_df_cleaned)

In [67]:
trained_model = train_model(train_df, val_df)

Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.
Epoch 1/10:   2%|█▊                                                                                                      | 19/1055 [00:52<04:48,  3.60it/s, loss=1.7162]Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7fbab22298b0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.8/dist-packages/torch/utils/data/dataloader.py", line 1479, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.8/dist-packages/torch/utils/data/dataloader.py", line 1462, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/python3.8/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
Epoch 1/10:   2%|██▎                                      

Epoch 1: TrainLoss=0.3225 | ValSMAPE=52.67%
✅ Saved new best model with SMAPE=52.67%


Epoch 2/10: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████| 1055/1055 [20:08<00:00,  1.15s/it, loss=0.1579]


Epoch 2: TrainLoss=0.1984 | ValSMAPE=50.53%
✅ Saved new best model with SMAPE=50.53%


Epoch 3/10: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████| 1055/1055 [24:57<00:00,  1.42s/it, loss=0.2120]


...

Epoch 6: TrainLoss=0.0575 | ValSMAPE=49.61%


Epoch 7/10: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████| 1055/1055 [20:30<00:00,  1.17s/it, loss=0.0384]


Epoch 7: TrainLoss=0.0432 | ValSMAPE=49.74%


Epoch 8/10:   0%|                                                                                                                              | 0/1055 [00:00<?, ?it/s]/usr/local/lib/python3.8/dist-packages/transformers/image_utils.py:216: DeprecationWarning: An exception was ignored while fetching the attribute `__array_interface__` from an object of type 'Image'.  With the exception of `AttributeError` NumPy will always raise this exception in the future.  Raise this deprecation warning to see the original exception. (Warning added NumPy 1.21)
  return np.array(img)
/usr/local/lib/python3.8/dist-packages/transformers/image_utils.py:216: DeprecationWarning: An exception was ignored while fetching the attribute `__array_interface__` from an object of type 'Image'.  With the exception of `AttributeError` NumPy will always raise this exception in the future.  Raise this deprecation warning to see the original exception. (Warning added NumPy 1.21)
  return np.array(img)
/usr/local/lib/pyt

KeyboardInterrupt: 

In [44]:
# Check types of all columns
print(train_df.dtypes)

# Check only numeric columns
numeric_cols = ["value", "ipq"]  # your numeric features
print(train_df[numeric_cols].dtypes)


sample_id                int64
price                  float64
image_path              object
value                  float64
ipq                    float64
                        ...   
unit_product_weight    float64
unit_sq                float64
unit_units             float64
unit_unità             float64
unit_unknown           float64
Length: 90, dtype: object
value    float64
ipq      float64
dtype: object


In [45]:
train_df.columns

Index(['sample_id', 'price', 'image_path', 'value', 'ipq', 'clean_text',
       'log_price', 'unit_1', 'unit_2', 'unit_20', 'unit_24', 'unit_7',
       'unit_8', 'unit_BOX', 'unit_Bag', 'unit_Bottle', 'unit_Box',
       'unit_Bucket', 'unit_CASE', 'unit_COUNT', 'unit_CT', 'unit_Can',
       'unit_Carton', 'unit_Comes', 'unit_Count', 'unit_Each', 'unit_FL',
       'unit_Fl', 'unit_Fluid', 'unit_Foot', 'unit_Gram', 'unit_Grams',
       'unit_Jar', 'unit_K', 'unit_LB', 'unit_Liters', 'unit_None', 'unit_OZ',
       'unit_Ounce', 'unit_Ounces', 'unit_Oz', 'unit_PACK', 'unit_Pack',
       'unit_Packs', 'unit_Paper', 'unit_Per', 'unit_Piece', 'unit_Pouch',
       'unit_Pound', 'unit_Pounds', 'unit_Sq', 'unit_Tea', 'unit_Ziplock',
       'unit_bag', 'unit_bottle', 'unit_bottles', 'unit_box', 'unit_can',
       'unit_capsule', 'unit_cm', 'unit_count', 'unit_ct', 'unit_each',
       'unit_fl', 'unit_fluid', 'unit_gr', 'unit_gram', 'unit_gramm',
       'unit_grams', 'unit_in', 'unit_kg', 'unit_lb

In [48]:
numeric_cols = ["value", "ipq"] + [col for col in train_df.columns if col.startswith("unit_")]
print(train_df[numeric_cols].dtypes.unique())

[dtype('float64')]


In [59]:
print(set(numeric_cols) - set(train_df.columns))  # Should be empty


set()


In [ ]:
test

In [71]:
test_final_df = prepare_inference_data(test_df_cleaned, encoder, scaler)

In [72]:
# Initialize the tokenizer and image processor with the same models
tokenizer = AutoTokenizer.from_pretrained(Config.TEXT_MODEL)
image_processor = AutoImageProcessor.from_pretrained(Config.IMAGE_MODEL_NAME)

# Initialize the text and image encoders
text_encoder = AutoModel.from_pretrained(Config.TEXT_MODEL)
img_encoder = AutoModel.from_pretrained(Config.IMAGE_MODEL_NAME)

Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.


In [74]:
# Re-create the model structure
final_model = MultiModalRegressor(text_encoder, img_encoder).to(Config.DEVICE)

# Define the path to your best saved model checkpoint
# **IMPORTANT**: Update the filename to match the best score you got (e.g., 48.39)
best_model_path = os.path.join(Config.CHECKPOINT_DIR, "best_model_48.39.pth")

print(f"Loading best model from: {best_model_path}")

# Load the saved weights into the model structure
final_model.load_state_dict(torch.load(best_model_path))
final_model.eval() # Set the model to evaluation mode (very important!)

# --- Step 4: Create the Test Dataset and DataLoader ---

# Get the list of all numeric and one-hot encoded columns
numeric_cols = ['value', 'ipq'] + [c for c in test_final_df.columns if c.startswith("unit_")]


Loading best model from: ./checkpoints_light/best_model_48.39.pth


In [75]:
# Create the dataset for the test data
test_ds = ProductDataset(
    test_final_df,
    tokenizer,
    image_processor,
    numeric_cols,
    is_train=False  # Set is_train to False
)

# Create a DataLoader to process the test data in batches
# A larger batch size can be used for inference
test_loader = torch.utils.data.DataLoader(
    test_ds,
    batch_size=Config.BATCH_SIZE * 2, # Often can double batch size for inference
    shuffle=False,
    num_workers=Config.NUM_WORKERS
)


In [76]:
# --- Step 5: Run Predictions ---

test_preds = []
print("\nGenerating predictions on the test set...")

# No need to calculate gradients for inference
with torch.no_grad():
    for batch in tqdm(test_loader):
        # Move the batch of data to the GPU
        batch = {k: v.to(Config.DEVICE) for k, v in batch.items()}

        # Get the model's predictions (they will be on the log scale)
        with autocast():
             preds_log = final_model(batch)

        # Move predictions to the CPU and store them
        test_preds.append(preds_log.cpu().squeeze().numpy())

# Concatenate predictions from all batches into a single array
final_predictions_log = np.concatenate(test_preds)


# --- Step 6: Format and Save the Submission File ---

# Convert predictions from log scale back to the original price scale
final_predictions = np.expm1(final_predictions_log)

# Create the submission DataFrame
submission_df = pd.DataFrame({
    'sample_id': test_final_df['sample_id'],
    'price': final_predictions
})

# Safety check: ensure no prices are negative
submission_df['price'] = submission_df['price'].clip(lower=0)

# Save to CSV
submission_df.to_csv('submission_48.39_model.csv', index=False)

print("\n✅ Submission file 'submission_dl_model.csv' created successfully!")
print("Here's a preview:")
print(submission_df.head())


Generating predictions on the test set...


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 586/586 [14:21<00:00,  1.47s/it]



✅ Submission file 'submission_dl_model.csv' created successfully!
Here's a preview:
   sample_id      price
0     100179  34.875000
1     245611  13.898438
2     146263  25.093750
3      95658   4.941406
4      36806  18.734375
